<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/audio_feature_extracor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sample = dataset[0]["audio"]
print(sample)

In [ ]:
result = pipe(sample)
print(result["text"])

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_voice(filename='recording.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Record';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.callback(reader.result);
        };
      };

      button.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          button.textContent = 'Stop Recording';
        } else {
          recorder.stop();
          button.textContent = 'Done!';
        }
      };

      return new Promise((resolve) => {
        window.callback = resolve;
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Run the function
audio_file = record_voice()
print(f"Saved as {audio_file}")

In [ ]:
from IPython.display import Audio
Audio(audio_file)

In [ ]:
generate_kwargs = {
    "language": "english",
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

# result = pipe(sample, return_timestamps=True)
result = pipe('recording.wav', generate_kwargs=args)
print(result["text"])

In [ ]:
import torch
import gc
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# 2. Load the native model components
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"
).to(device)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define your Chunking and Stride rules
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

# This structure will hold your explicit chunk-wise data
chunked_results = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Processing explicit chunks manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    # Calculate the global clock positioning for metadata records
    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftover audio shards
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # FIX 1: Generate BOTH input_features and attention_mask
    inputs = processor(chunk, sampling_rate=sr, return_attention_mask=True, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)
    attention_mask = inputs.attention_mask.to(device)

    # Generate text & word timestamps for THIS SPECIFIC CHUNK ONLY
    with torch.no_grad():
        # FIX 2: Set return_dict_in_generate=True so we can safely unpack outputs
        outputs = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            return_timestamps=True, # 'word' for word-level timestamps
            return_dict_in_generate=True, # Wraps outputs in a safe dictionary structure
            temperature=0.0
        )

    # FIX 3: Unpack the generated text token ids cleanly
    predicted_ids = outputs["sequences"]
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Append the structured result for this explicit block
    chunked_results.append({
        "chunk_index": len(chunked_results),
        "global_window_seconds": (global_start_time, global_end_time),
        "text": transcription.strip(),
    })

    print(f"Processed Chunk {chunked_results[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

# 6. Inspect your isolated chunk-wise data structure
print("\n--- VIEW OF MANUALLY SEPARATED CHUNKS ---")
import pprint
pprint.pprint(chunked_results)

In [ ]:
#@title Part 1: Explicit Audio Chunking with WavLM Embeddings
import torch
import gc
import numpy as np
from transformers import AutoFeatureExtractor, WavLMModel
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# Note: WavLM is typically stable in float32; float16 can be used on modern GPUs
torch_dtype = torch.float32

model_id = "microsoft/wavlm-base-plus"

# 2. Load WavLM Native Components
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = WavLMModel.from_pretrained(model_id).to(device, dtype=torch_dtype)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define Chunking and Stride rules (matching your original spec)
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

chunked_embeddings = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Extracting acoustic chunk embeddings manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftovers
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # Extract structural acoustic features
    inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt")
    input_values = inputs.input_values.to(device, dtype=torch_dtype)

    with torch.no_grad():
        outputs = model(input_values)
        # Sequence shape: [batch, sequence_length, 768]
        last_hidden_states = outputs.last_hidden_state

        # Mean Pooling: Collapse time dimension to catch holistic properties of the window
        mean_pooled = torch.mean(last_hidden_states, dim=1).squeeze()
        embedding = mean_pooled.cpu().numpy()

    chunked_embeddings.append({
        "chunk_index": len(chunked_embeddings),
        "global_window_seconds": (global_start_time, global_end_time),
        "embedding": embedding # 768-dimensional float array
    })

    print(f"Processed Chunk {chunked_embeddings[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s | Shape: {embedding.shape}")

# To represent the entire track as a single, holistic master vector:
all_vectors = [c["embedding"] for c in chunked_embeddings]
master_audio_embedding = np.mean(all_vectors, axis=0)
print(f"\nFinal global track embedding generated with shape: {master_audio_embedding.shape}")

In [ ]:
#@title Part 2: Measuring Similarity (Qualitative vs. Quantitative)
import torch.nn.functional as F

# Helper function to extract a pooled embedding vector from raw audio
def get_holistic_embedding(audio_data, sampling_rate=16000):
    inputs = feature_extractor(audio_data, sampling_rate=sampling_rate, return_tensors="pt")
    inputs = inputs.input_values.to(device, dtype=torch_dtype)
    with torch.no_grad():
        outputs = model(inputs)
        return torch.mean(outputs.last_hidden_state, dim=1) # Keeps batch dim for F.cosine_similarity

# --- Scenario A: Qualitative Similarity Example ---
# Two audio slices from the exact same speaker, recording environment, and background noise levels,
# but they are saying entirely different sentences.
audio_speaker1_phraseA = audio_array[0 : 10 * sr]             # First 10 seconds
audio_speaker1_phraseB = audio_array[15 * sr : 25 * sr]       # A completely different 10 seconds

emb_qual_1 = get_holistic_embedding(audio_speaker1_phraseA)
emb_qual_2 = get_holistic_embedding(audio_speaker1_phraseB)

qualitative_similarity = F.cosine_similarity(emb_qual_1, emb_qual_2).item()


# --- Scenario B: Quantitative Similarity Example ---
# Two audio clips with matching cadence properties. For demonstration, we simulate
# a quantitative variation (like an exact pitch shift or clean speed alteration)
# to show how structural audio manipulation preserves high quantitative feature overlaps.
audio_base = audio_array[0 : 15 * sr]

# Simulate a clean quantitative shift (e.g., applying a minor pitch modulation natively)
# For code safety without extra external dependencies, we use a basic array operation
audio_pitched = np.ascontiguousarray(audio_base * 0.95)

emb_quant_1 = get_holistic_embedding(audio_base)
emb_quant_2 = get_holistic_embedding(audio_pitched)

quantitative_similarity = F.cosine_similarity(emb_quant_1, emb_quant_2).item()


# --- Print Metric Diagnostics ---
print("\n--- EMBEDDING SIMILARITY ANALYSIS ---")
print(f"Qualitative Match (Same Speaker / Context, Different Words): {qualitative_similarity:.4f}")
print(f"Quantitative Match (Same Pacing / Signal Structure Alteration): {quantitative_similarity:.4f}")

In [ ]:
#@title DLAI Suggestion for Interview Rating pipeline
import torch
import numpy as np
from transformers import pipeline

# =====================================================================
# STEP 1: AUDIO TRANSCRIPTION & TIME ANALYTICS (Using Whisper)
# =====================================================================

# Initialize Whisper pipeline with chunking and timestamp options enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device
)

def analyze_interview_audio(audio_path_or_array):
    print("Processing audio with Whisper...")
    # Retrieve transcription with word-level timestamps
    result = pipe(
        audio_path_or_array,
        return_timestamps="word"
    )

    text = result["text"]
    chunks = result["chunks"]

    # Programmatically calculate delivery metrics
    total_words = len([c for c in chunks if "text" in c])

    # Calculate silences/pauses (gaps between words greater than 1.5 seconds)
    pauses = 0
    for i in range(len(chunks) - 1):
        end_current = chunks[i]["timestamp"][1]
        start_next = chunks[i+1]["timestamp"][0]
        if end_current is not None and start_next is not None:
            if (start_next - end_current) > 1.5:
                pauses += 1

    total_duration = chunks[-1]["timestamp"][1] if chunks else 1.0
    words_per_minute = (total_words / total_duration) * 60

    return {
        "transcript": text,
        "metrics": {
            "words_per_minute": round(words_per_minute, 1),
            "long_pauses_count": pauses,
            "total_duration_seconds": round(total_duration, 2)
        }
    }

# Simulating processing on a dummy sample
# sample_audio = "candidate_answer.mp3"
# audio_analysis = analyze_interview_audio(sample_audio)

# Mocked output for the sake of the structural demonstration:
audio_analysis = {
    "transcript": "A REST API is stateless... um, meaning that the server does not store any session data about the client. Every request must, uh, contain all the information needed.",
    "metrics": {
        "words_per_minute": 110.5,
        "long_pauses_count": 2,
        "total_duration_seconds": 15.4
    }
}

print("\n--- Audio Analytics Extracted ---")
print(audio_analysis)

# =====================================================================
# STEP 2: KNOWLEDGE VERIFICATION (LLM-as-a-Judge Prompt)
# =====================================================================

# This is how you would construct your prompt to a foundational LLM
# to keep it grounded, objective, and outputting structured JSON.

reference_answer_from_rag = """
A REST API must be stateless. The server should not store any context or session data about the client.
Each individual request from a client must contain all necessary information and authentication details to understand and process it.
"""

llm_judge_prompt = f"""
You are an expert technical interviewer acting as a strict, objective grading judge.
Analyze the candidate's transcript against the Reference Answer retrieved from our Knowledge Base.

[Reference Answer]
{reference_answer_from_rag}

[Candidate Transcript]
{audio_analysis['transcript']}

[Grading Instructions]
1. Evaluate if the logical sequence matches and if the fundamental points are fully covered.
2. To prevent hallucination or being overly generous, you MUST extract a direct quote from the Candidate Transcript to prove a point was met.
3. Ignore minor verbal fillers like "um" or "uh" (focus entirely on semantic correctness).

Provide your final assessment strictly in the following JSON format:
{{
  "technical_accuracy_score": <int from 0 to 100>,
  "points_covered": [
     {{"point": "Statelessness criteria", "status": "Met/Unmet", "justifying_quote": "string or null"}}
  ],
  "logical_progression_rating": "Excellent/Fair/Poor",
  "constructive_feedback": "string"
}}
"""

print("\n--- Generated LLM-as-a-Judge System Prompt ---")
print(llm_judge_prompt)